# D502 Capstone Project

This Jupyter notebook is for Joan Lambert's D502 Capstone project, The Effect of State Investment in Education.

## 1. Set up the Environment

In [ ]:
# Import libraries for EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print('✓ All required packages imported successfully!')

## 2. Load the Data

### Data Sources


This project uses data from the following sources:
1.	NCES Digest of Education Statistics, Table 219.46: Public high school graduation rates (ACGR)
2.	NCES Digest of Education Statistics, Table 236.65: Per-pupil expenditure
3.	US Census Bureau American Community Survey, Table B17001: Poverty status
4.	US Census Bureau American Community Survey, Table B19013: Median household income

### Data Preparation

I manually downloaded the NCES data in Excel file format. I then merged the four sources into a single panel dataset using the following data acquisition and cleaning scripts:
- `src/pull_acs_data.py` - Census Bureau API data retrieval
- `src/process_nces_data.py` - NCES Excel file processing
- `src/merge_datasets.py` - Dataset merging
- `src/adjust_inflation.py` - CPI adjustment to 2019 dollars

The final dataset contains 500 state-year observations (50 states × 10 years). Nine graduation values are missing from the original source.

In [ ]:
# Load final cleaned and merged dataset
df = pd.read_csv('data/clean/final_panel.csv')
print(f"Loaded {len(df)} observations")
df.head()

## 3. Conduct Exploratory Data Analysis

### 3.1 Generate Descriptive Statistics

In [ ]:
# Basic dataset info
print("Dataset Overview")
print("="*70)
print(f"Shape: {df.shape}")
print(f"Years: {sorted(df['year'].unique())}")
print(f"States: {df['state_name'].nunique()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Descriptive statistics for key variables
print("\nDescriptive Statistics (all values in 2019 dollars)")
print("="*70)
desc_stats = df[['graduation_rate', 'per_pupil_spending_2019', 
                 'median_hh_income_2019', 'poverty_rate']].describe()
print(desc_stats.round(2))

#### Interpretation

The descriptive statistics indicate...

### 3.2 Create a Correlation Matrix with a Heatmap

In [ ]:
# Select variables for correlation analysis
corr_vars = ['graduation_rate', 'per_pupil_spending_2019', 
             'median_hh_income_2019', 'poverty_rate']

# Calculate correlation matrix
corr_matrix = df[corr_vars].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.3f')
plt.title('Correlation Matrix of Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(corr_matrix.round(3))

#### Interpretation

The correlation matrix indicates...

### 3.3 VIF for Multicollinearity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Prepare data for VIF (drop rows with missing graduation_rate)
df_complete = df.dropna(subset=['graduation_rate'])

# Select independent variables
X = df_complete[['per_pupil_spending_2019', 'median_hh_income_2019', 'poverty_rate']]

# Calculate VIF for each variable
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print("\nVariance Inflation Factors (VIF)")
print("="*70)
print(vif_data)
print("\nInterpretation: VIF < 5 indicates acceptable multicollinearity")
print("                VIF 5-10 indicates moderate multicollinearity")
print("                VIF > 10 indicates high multicollinearity")

#### Interpretation

The VIF indicates...

### 3.4 Time Series Visualization

In [ ]:
# Average graduation rate and spending over time
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot graduation rate
color = 'tab:blue'
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Graduation Rate (%)', color=color, fontsize=12)
yearly_grad = df.groupby('year')['graduation_rate'].mean()
ax1.plot(yearly_grad.index, yearly_grad.values, color=color, marker='o', linewidth=2, label='Graduation Rate')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Plot spending on secondary axis
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Per-Pupil Spending (2019 $)', color=color, fontsize=12)
yearly_spend = df.groupby('year')['per_pupil_spending_2019'].mean()
ax2.plot(yearly_spend.index, yearly_spend.values, color=color, marker='s', linewidth=2, label='Per-Pupil Spending')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('National Trends: Graduation Rate and Per-Pupil Spending (2010-2019)', 
          fontsize=14, fontweight='bold', pad=20)
fig.tight_layout()
plt.show()

#### Interpretation

The time series visualization indicates...

### 3.5 Scatter Plot - Spending vs Graduation Rate

In [ ]:
# Create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df['per_pupil_spending_2019'], df['graduation_rate'], alpha=0.5)

# Add trend line
z = np.polyfit(df['per_pupil_spending_2019'].dropna(), 
               df.dropna(subset=['per_pupil_spending_2019', 'graduation_rate'])['graduation_rate'], 1)
p = np.poly1d(z)
plt.plot(df['per_pupil_spending_2019'].sort_values(), 
         p(df['per_pupil_spending_2019'].sort_values()), 
         "r--", alpha=0.8, linewidth=2, label=f'Trend line: y={z[0]:.4f}x+{z[1]:.2f}')

plt.xlabel('Per-Pupil Spending (2019 $)', fontsize=12)
plt.ylabel('Graduation Rate (%)', fontsize=12)
plt.title('Relationship Between Per-Pupil Spending and Graduation Rates\n(All State-Year Observations, 2010-2019)', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Interpretation

The scatter plot indicates...

### 3.6 Distribution Plots

In [ ]:
# Distribution of key variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Graduation Rate
axes[0, 0].hist(df['graduation_rate'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Graduation Rate (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Graduation Rates')
axes[0, 0].axvline(df['graduation_rate'].mean(), color='red', linestyle='--', label=f'Mean: {df["graduation_rate"].mean():.1f}%')
axes[0, 0].legend()

# Per-Pupil Spending
axes[0, 1].hist(df['per_pupil_spending_2019'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Per-Pupil Spending (2019 $)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Per-Pupil Spending')
axes[0, 1].axvline(df['per_pupil_spending_2019'].mean(), color='red', linestyle='--', label=f'Mean: ${df["per_pupil_spending_2019"].mean():.0f}')
axes[0, 1].legend()

# Poverty Rate
axes[1, 0].hist(df['poverty_rate'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('Poverty Rate (%)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Poverty Rates')
axes[1, 0].axvline(df['poverty_rate'].mean(), color='red', linestyle='--', label=f'Mean: {df["poverty_rate"].mean():.1f}%')
axes[1, 0].legend()

# Median Household Income
axes[1, 1].hist(df['median_hh_income_2019'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_xlabel('Median Household Income (2019 $)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Median Household Income')
axes[1, 1].axvline(df['median_hh_income_2019'].mean(), color='red', linestyle='--', label=f'Mean: ${df["median_hh_income_2019"].mean():.0f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

#### Interpretation

The distribution plots indicate...